In [ ]:
import h5py
import numpy as np
import re
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt

In [1]:
# ------------------------------------------------------------------
# 1. List of files you want to scan (each with some MeV cut(s))
# ------------------------------------------------------------------
filepaths = [
    '/Users/justin/code/trajectum/cs_analysis/src/nosmash/7GeV_FinalResult1_NoSmash_80000_200PJ.h5',
    '/Users/justin/code/trajectum/cs_analysis/src/nosmash/19GeV_FinalResult1_NoSmash_80000_400PJ.h5',
    '/Users/justin/code/trajectum/cs_analysis/src/nosmash/27GeV_FinalResult1_NoSmash_80000_400PJ.h5'
    '/Users/justin/code/trajectum/cs_analysis/src/nosmash/54GeV_FinalResult1_NoSmash_80000_500PJ.h5'
    '/Users/justin/code/trajectum/cs_analysis/src/nosmash/200GeV_FinalResult1_NoSmash_80000_500J.h5'
]


In [ ]:
#
# This is the helper fuction which retrieves the relevant file information 
#

def get_fluctuation_curves_from_file(filepath):
    curves = []  # each element: (mev_label, centrality, y, yerr)

    with h5py.File(filepath, "r") as hdf:
        # centrality axis (common to all cuts in this file)
        centrality = hdf["centrality"][:].squeeze()

        # find all STARTPCxxxMeV groups under meanptcharged
        if "meanptcharged" not in hdf:
            raise KeyError(f"'meanptcharged' group not found in {filepath}")

        startpc_groups = [
            key for key in hdf["meanptcharged"].keys()
            if key.startswith("STARTPC")
        ]

        for grp in startpc_groups:
            # parse the MeV value from something like "STARTPC200MeV"
            m = re.search(r"(\d+)MeV", grp)
            mev_label = m.group(1) if m else grp  # "200" from "STARTPC200MeV"

            mean_base = f"meanptcharged/{grp}/centralitybinned"
            fluc_base = f"ptfluctuationscharged/{grp}/centralitybinned"

            # mean pT charged
            meanpTcharged_values = hdf[f"{mean_base}/values"][:].squeeze()
            meanpTcharged_uerr   = hdf[f"{mean_base}/uppererrors"][:].squeeze()
            meanpTcharged_lerr   = hdf[f"{mean_base}/lowererrors"][:].squeeze()
            meanpTcharged_symerr = 0.5 * (meanpTcharged_uerr + meanpTcharged_lerr)

            # delta pT
            deltapT_values = hdf[f"{fluc_base}/values"][:].squeeze()
            deltapT_uerr   = hdf[f"{fluc_base}/uppererrors"][:].squeeze()
            deltapT_lerr   = hdf[f"{fluc_base}/lowererrors"][:].squeeze()
            deltapT_symerr = 0.5 * (deltapT_uerr + deltapT_lerr)

            # δpT / ⟨pT⟩ in percent
            # normalizedfluctuation = (deltapT_values / meanpTcharged_values) * 100.0
            normalizedfluctuation = (deltapT_values / 1) * 100.0

            # error propagation
            with np.errstate(divide='ignore', invalid='ignore'):
                ratio_err = normalizedfluctuation * np.sqrt(
                    (deltapT_symerr / deltapT_values)**2 +
                    (meanpTcharged_symerr / meanpTcharged_values)**2
                )

            # clean bad points
            mask = (
               np.isfinite(normalizedfluctuation) &
               np.isfinite(ratio_err) &
               (ratio_err >= 0)
            )

            #print(normalizedfluctuation)
            #print(ratio_err)

            curves.append((
                mev_label,
                centrality[mask],
                normalizedfluctuation[mask],
                ratio_err[mask],
            ))

    return curves
